In [39]:
import pickle
import os
import re
import random
from fuzzywuzzy import process, fuzz
from sklearn.metrics.pairwise import cosine_similarity
import torch


In [ ]:
# ==========================================
# USER PROFILE (CBR RETAIN PHASE)
# ==========================================
class UserProfile:
    def __init__(self):
        self.weights = {
            "genre": 0.4,
            "studio": 0.2,
            "vibe": 0.4 
        }
        self.learning_rate = 0.05
        self.max_weight = 0.85
        self.min_weight = 0.1

    def learn(self, interaction_type):
        if interaction_type == "genre_search":
            self.weights["genre"] = min(self.max_weight, self.weights["genre"] + self.learning_rate)
            self.weights["vibe"] = max(self.min_weight, self.weights["vibe"] - (self.learning_rate / 2))
        elif interaction_type == "specific_title":
            self.weights["vibe"] = min(self.max_weight, self.weights["vibe"] + self.learning_rate)
            self.weights["genre"] = max(self.min_weight, self.weights["genre"] - (self.learning_rate / 2))
        
        # Normalize
        total = sum(self.weights.values())
        for k in self.weights: self.weights[k] /= total

# ==========================================
# EXPLAINABLE CHATBOT
# ==========================================
class AnimeChatbot:
    def __init__(self, model, entity_to_id, anime_data, label_dict):
        self.model = model
        self.entity_to_id = entity_to_id
        self.anime_data = anime_data
        self.id_to_name = label_dict
        self.valid_ids = set(anime_data.keys())
        self.user = UserProfile()

        # Embeddings
        self.model.eval()
        with torch.no_grad():
            self.all_embeddings = self.model.entity_representations[0](indices=None).detach().cpu().numpy()

        # Search List
        self.name_to_id = {}
        for entity_id in self.entity_to_id.keys():
            if entity_id in self.valid_ids:
                readable_name = self.id_to_name.get(entity_id, entity_id)
                self.name_to_id[readable_name] = entity_id
        self.searchable_names = list(self.name_to_id.keys())
        
        # Genre Map
        self.genre_map = {}
        self.genre_to_anime_list = {} 
        for aid, feats in self.anime_data.items():
            for g_uri in feats["G"]:
                raw = g_uri.split("/")[-1].split("#")[-1].lower()

                # NORMALIZACIÓN REAL:
                short_name = (
                    raw.replace("__anime", "")
                    .replace("_anime", "")
                    .replace("anime", "")
                    .replace("_", " ")
                    .strip()
                )

                # mapping limpio
                self.genre_map[short_name] = g_uri

                # registrar anime por género
                if short_name not in self.genre_to_anime_list:
                    self.genre_to_anime_list[short_name] = []
                self.genre_to_anime_list[short_name].append(aid)

        # Configs
        self.aliases = {
            "demon slayer": "Kimetsu no Yaiba",
            "attack on titan": "Shingeki no Kyojin",
            "my hero academia": "Boku no Hero Academia"
        }
        self.filler_patterns = [
            r"i like", r"i love", r"i watched", r"im watching", r"recommend", 
            r"looking for", r"similar to", r"anime like", r"find me", r"show me",
            r"anime", r"animes", r"aniies", r"series", r"show"
        ]

    def _extract_title_and_genre(self, sentence):
        clean_sent = sentence.lower()
        found_genre_uri = None
        found_genre_name = None

        for g_name, g_uri in self.genre_map.items():
            pattern = r'\b' + re.escape(g_name) + r'\b'
            if re.search(pattern, clean_sent):
                found_genre_uri = g_uri
                found_genre_name = g_name
                clean_sent = clean_sent.replace(g_name, "")
                break 

        clean_sent = re.sub(r'[^\w\s]', '', clean_sent)
        for pattern in self.filler_patterns:
            clean_sent = re.sub(pattern, '', clean_sent)

        if found_genre_name == "":
            found_genre_name = None

        print("DETECTED GENRE:", found_genre_name)
        print("CLEANED TEXT:", clean_sent)
        
        return clean_sent.strip(), found_genre_uri, found_genre_name

    def recommend(self, user_input):
        extracted_query, boost_genre_uri, boost_genre_name = self._extract_title_and_genre(user_input)
        
        # 1. Genre Only
        if boost_genre_name and (len(extracted_query) < 4 or extracted_query in ["i like", "i love", "i really like", "looking for", ""]):
            self.user.learn("genre_search")
            return self._recommend_genre_only(boost_genre_name)

        # 2. Specific Title
        if len(extracted_query) < 2: extracted_query = user_input
        if extracted_query in self.aliases: extracted_query = self.aliases[extracted_query]

        best_match, score = process.extractOne(
            extracted_query, 
            self.searchable_names, 
            scorer=fuzz.token_set_ratio
        )

        if score < 60: 
            return f"I couldn't identify '{extracted_query}'. Did you mean: {best_match}?"

        self.user.learn("specific_title")
        target_id = self.name_to_id[best_match]
        
        # === CALL CALCULATE ===
        recs = self._calculate(target_id, boost_genre_uri)

        # === FORMAT OUTPUT WITH EXPLANATION ===
        response = f"Found: **{best_match}**\n"
        if boost_genre_name:
            response += f"Filtering for: **{boost_genre_name.capitalize()}**\n"
        
        response += "Top recommendations with Explanations:\n"
        response += "="*40 + "\n"
        
        for i, (name, final_score, explanation) in enumerate(recs, 1):
            response += f"{i}. **{name}** ({int(final_score*100)}%)\n"
            response += f"   • {explanation['reason']}\n"
            response += f"   • [Math]: Genre: {explanation['s_g']:.2f} | Studio: {explanation['s_e']:.0f} | Vector: {explanation['s_v']:.2f}\n"
        
        return response

    def _recommend_genre_only(self, genre_name):
        candidates = self.genre_to_anime_list.get(genre_name, [])
        if not candidates: return f"No anime found for {genre_name}."
        selected = random.sample(candidates, min(5, len(candidates)))
        
        response = f"Here are some **{genre_name.capitalize()}** anime:\n"
        for i, aid in enumerate(selected, 1):
            name = self.id_to_name.get(aid, aid).replace("_", " ")
            response += f"{i}. **{name}**\n"
        return response

    def _calculate(self, target, boost_genre=None, top_k=5):
        target_idx = self.entity_to_id[target]
        target_emb = self.all_embeddings[target_idx].reshape(1, -1)
        t_feats = self.anime_data.get(target)
        
        sim_scores = cosine_similarity(target_emb, self.all_embeddings)[0]

        scores = []
        target_readable = self.id_to_name.get(target, target).replace("_", " ").lower()
        
        w_genre = self.user.weights["genre"]
        w_studio = self.user.weights["studio"]
        w_vibe = self.user.weights["vibe"]

        for name in self.anime_data:
            if name == target: continue
            if name not in self.entity_to_id: continue

            readable = self.id_to_name.get(name, name).replace("_", " ")
            if "movie" in readable.lower(): continue 
            if target_readable.split()[0] in readable.lower(): continue
            if readable.lower() == target_readable: continue

            # 1. TransR Score
            idx = self.entity_to_id[name]
            s_transr = (sim_scores[idx] + 1) / 2

            # 2. Content Scores
            feats = self.anime_data[name]
            
            # Intersection for explanation
            shared_genres = t_feats["G"].intersection(feats["G"])
            shared_studio = t_feats["E"].intersection(feats["E"])
            
            u_g = len(t_feats["G"].union(feats["G"]))
            s_g = len(shared_genres) / u_g if u_g > 0 else 0
            s_e = 1.0 if len(shared_studio) > 0 else 0.0
            
            # 3. Final Score
            final = (w_genre * s_g) + (w_studio * s_e) + (w_vibe * s_transr)

            # Boost
            boosted = False
            if boost_genre and boost_genre in feats["G"]:
                final += 0.2
                if final > 1.0: final = 1.0
                boosted = True
            
            # 4. GENERATE EXPLANATION
            reasons = []
            
            # Explain Genre
            clean_genres = [g.split("/")[-1].split("#")[-1] for g in shared_genres]
            if len(clean_genres) > 0:
                reasons.append(f"Shares {len(clean_genres)} genres ({', '.join(clean_genres[:3])})")
            
            # Explain Studio
            if s_e > 0:
                clean_studio = list(shared_studio)[0].split("/")[-1].split("#")[-1].replace("_", " ")
                reasons.append(f"Same Studio ({clean_studio})")
            
            # Explain Vector
            if s_transr > 0.8:
                reasons.append("High semantic similarity (Plot/Themes)")
            elif s_transr > 0.6:
                reasons.append("Similar atmosphere")
            
            if boosted:
                reasons.append(f"Boosted by user preference (+20%)")

            explanation = {
                "reason": ", ".join(reasons),
                "s_g": s_g,
                "s_e": s_e,
                "s_v": s_transr
            }

            scores.append((readable, final, explanation))

        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [36]:
def load_system(folder_path):
    print(f"Loading system from '{folder_path}'...")
    
    # Path to the model file
    model_path = os.path.join(folder_path, "model.pt")
    
    # 1. Load Model
    # We set weights_only=False because PyKEEN saves the whole class structure, 
    # not just the weights. Since you built this model, it is safe.
    model = torch.load(
        model_path, 
        map_location=torch.device('cpu'), 
        weights_only=False 
    )
    
    # 2. Load Triples Factory (For ID mappings)
    with open(os.path.join(folder_path, "triples_factory.pkl"), "rb") as f:
        triples_factory = pickle.load(f)
        
    # 3. Load Data Package (Labels and Features)
    with open(os.path.join(folder_path, "data_package.pkl"), "rb") as f:
        data_pkg = pickle.load(f)
    
    print("System loaded successfully.")
    
    # Initialize Chatbot
    return AnimeChatbot(
        model=model,
        entity_to_id=triples_factory.entity_to_id,
        anime_data=data_pkg["anime_data"],
        label_dict=data_pkg["id_to_label"]
    )

In [ ]:
folder_name = "anime_recommender_final" 
    
try:
    bot = load_system(folder_name)
    
    print("\n--- Ready to Chat ---")
    print(bot.recommend("I really like action anime"))
    print(bot.recommend("Show me adventure anime"))
    print(bot.recommend("I want something like One Piece"))

        
except FileNotFoundError:
    print(f"Error: Could not find folder '{folder_name}'. Make sure you extracted the zip file there.")

Loading system from 'anime_recommender_final'...
System loaded successfully.

--- Ready to Chat ---
DETECTED GENRE: action
CLEANED TEXT:   
Here are some **Action** anime:
1. **Drifters**
2. **Magical Girl Lyrical Nanoha StrikerS**
3. **Cinderella Boy**
4. **Naruto The Lost Story  Mission Protect the Waterfall Village**
5. **Tokyo Babylon**

DETECTED GENRE: adventure
CLEANED TEXT:   
Here are some **Adventure** anime:
1. **Lupin III The Secret of Twilight Gemini**
2. **Sora o Miageru Shōjo no Hitomi ni Utsuru Sekai**
3. **AniKuri15**
4. **Digimon Our War Game**
5. **Bleach**

DETECTED GENRE: None
CLEANED TEXT: i want something like one piece
Found: **One Piece**
Top recommendations with Explanations:
1. **Shaman King** (55%)
   • Shares 3 genres (comedy__anime, action__anime, adventure__anime), Similar atmosphere
   • [Math]: Genre: 0.60 | Studio: 0 | Vector: 0.79
2. **Miami Guns** (55%)
   • Shares 3 genres (comedy__anime, action__anime, adventure__anime), Similar atmosphere
   • [Mat